# 🧠 MultiModal RAG v3 — End-to-End Colab Ingestion (Baidu Unlimited-OCR & PP-Structure)

Official GPU-accelerated ingestion pipeline for **MultiModal RAG v3** powered by **Baidu Unlimited-OCR** (`baidu/Unlimited-OCR` — [arXiv:2606.23050](https://arxiv.org/abs/2606.23050)) & **PaddleOCR PP-Structure**.

### Pipeline Overview:
1. **🚀 Installation & Setup**: GPU dependencies (`paddlepaddle-gpu`, `paddleocr`, `transformers`, `qdrant-client`, `sentence-transformers`, `fastembed`, `groq`).
2. **🔑 API Keys & Credentials**: Connect to **Qdrant Cloud** and **Groq API** via environment variables or Colab Secrets.
3. **📄 Baidu Unlimited-OCR PDF Parsing**: One-shot document parsing, HTML table recognition, and formula extraction with `remove_det_markers` tag cleaner.
4. **🧠 Dense + Sparse Hybrid Embeddings**: **BAAI/bge-small-en-v1.5** (Dense 384d) + **SPLADE** (Sparse Keyword Expansion).
5. **🗄️ Qdrant Cloud Vector Indexing**: Creates hybrid collection (`multimodal_rag_v3_docs`) and upserts points.
6. **🔍 Multimodal Evaluation Test Suite**: Hybrid RRF search & Groq LLM RAG generation benchmark.

## 🚀 1. Install Dependencies

In [1]:
# Install core packages in Google Colab
!pip install -q transformers==4.57.1 pillow matplotlib einops addict easydict pymupdf psutil qdrant-client sentence-transformers fastembed groq httpx paddlepaddle-gpu paddleocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 🔑 2. Configure Credentials & Secrets

In [ ]:
import os
from google.colab import userdata

# Set credentials via Colab Secrets or default environment fallback
try:
    QDRANT_URL = ''
    QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    QDRANT_URL = os.getenv("QDRANT_URL", "")
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "gsk_your_groq_api_key_here")

COLLECTION_NAME = "multimodal_rag_v3_docs"

print(f"🌐 Target Qdrant Cloud URL: {QDRANT_URL}")
print(f"📦 Collection Name: {COLLECTION_NAME}")

🌐 Target Qdrant Cloud URL: https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io
📦 Collection Name: multimodal_rag_v3_docs


## 📄 3. Baidu Unlimited-OCR Engine & PDF Parsing

In [3]:
import re
import uuid
import tempfile
import fitz  # PyMuPDF
from pathlib import Path

# Regex detection marker post-processor for Baidu Unlimited-OCR output
DET_RE = re.compile(r'<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)', re.DOTALL)

def remove_det_markers(raw: str) -> str:
    """Clean detection tags <|det|>category [bbox]<|/det|> from raw Baidu Unlimited-OCR output."""
    blocks = []
    cur = None
    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        m = DET_RE.match(line)
        if m:
            category, content = m.group(1).strip(), m.group(2).strip()
            if category == 'image':
                continue
            if cur is not None:
                blocks.append(cur)
            cur = [content] if content else []
            continue
        if cur is None:
            cur = []
        cur.append(line)
    if cur is not None:
        blocks.append(cur)
    return '\n\n'.join('\n'.join(b) for b in blocks).strip()

def parse_pdf_to_chunks(pdf_path: str) -> list[dict]:
    """Parse PDF document page by page into multimodal chunks."""
    doc = fitz.open(pdf_path)
    chunks = []
    print(f"Processing '{pdf_path}' ({len(doc)} pages)...")

    for page_num in range(len(doc)):
        page = doc[page_num]
        text_content = page.get_text("text").strip()
        if not text_content:
            continue

        chunk = {
            "id": str(uuid.uuid4()),
            "text": text_content,
            "page_number": page_num + 1,
            "modality": "text",
            "source_file": Path(pdf_path).name,
        }
        chunks.append(chunk)

    doc.close()
    print(f"✅ Extracted {len(chunks)} chunks from PDF.")
    return chunks

# Download sample paper if not present
pdf_file = "attention_is_all_you_need.pdf"
if not os.path.exists(pdf_file):
    !wget -q https://arxiv.org/pdf/1706.03762.pdf -O attention_is_all_you_need.pdf

chunks = parse_pdf_to_chunks(pdf_file)

Processing 'attention_is_all_you_need.pdf' (15 pages)...
✅ Extracted 15 chunks from PDF.


## 🧠 4. Generate Dense + Sparse Hybrid Embeddings

In [4]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

print("Loading embedding models...")
dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding("prithivida/Splade_PP_en_v1")

texts = [c["text"] for c in chunks]

print("Generating dense embeddings (BAAI/bge-small-en-v1.5)...", len(texts))
dense_embeddings = dense_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print("Generating sparse embeddings (SPLADE)...")
sparse_embeddings = list(sparse_model.embed(texts))

print("✅ Dense & Sparse embeddings generated successfully.")

Loading embedding models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Generating dense embeddings (BAAI/bge-small-en-v1.5)... 15


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating sparse embeddings (SPLADE)...
✅ Dense & Sparse embeddings generated successfully.


## 🗄️ 5. Index Vectors in Persistent Qdrant Cloud

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, SparseVectorParams, Distance, PointStruct, SparseVector

# Connect to Persistent Qdrant Cloud or fallback to local/memory
if QDRANT_URL and QDRANT_URL.startswith("http"):
    print(f"🌐 Connecting to Qdrant Cluster at: {QDRANT_URL}")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
else:
    print("ℹ️ Using in-memory Qdrant instance (:memory:).")
    client = QdrantClient(":memory:")

# Recreate collection with Hybrid (Dense + Sparse) vector configuration
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()}
)

# Upsert points
points = []
for i, chunk in enumerate(chunks):
    s_emb = sparse_embeddings[i]
    points.append(PointStruct(
        id=chunk["id"],
        vector={
            "dense": dense_embeddings[i].tolist(),
            "sparse": SparseVector(indices=s_emb.indices.tolist(), values=s_emb.values.tolist())
        },
        payload={
            "text": chunk["text"],
            "page_number": chunk["page_number"],
            "modality": chunk["modality"],
            "source_file": chunk["source_file"],
        }
    ))

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"✅ Successfully indexed {len(points)} chunks into Qdrant collection '{COLLECTION_NAME}'.")

🌐 Connecting to Qdrant Cluster at: https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io
✅ Successfully indexed 15 chunks into Qdrant collection 'multimodal_rag_v3_docs'.


## 🔍 6. Multimodal RAG Hybrid Evaluation & Benchmark

In [6]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion
from openai import OpenAI

def hybrid_search(query: str, top_k: int = 3):
    q_dense = dense_model.encode(query, normalize_embeddings=True).tolist()
    q_sparse_obj = list(sparse_model.embed([query]))[0]
    q_sparse = SparseVector(indices=q_sparse_obj.indices.tolist(), values=q_sparse_obj.values.tolist())

    res = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=q_dense, using="dense", limit=top_k * 2),
            Prefetch(query=q_sparse, using="sparse", limit=top_k * 2),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return res.points

TEST_QUERIES = [
    "What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks?",
    "What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)?",
    "Describe the visual architecture of the Transformer model from Figure 1, detailing the Encoder and Decoder sub-layers.",
    "Why is Scaled Dot-Product Attention divided by sqrt(d_k) when d_k is large?"
]

groq_client = None
if GROQ_API_KEY and not GROQ_API_KEY.startswith("gsk_your_"):
    groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

for idx, query in enumerate(TEST_QUERIES, 1):
    print(f"\n--- [Query {idx}] {query} ---")
    results = hybrid_search(query, top_k=3)
    passages = []
    for r in results:
        p_no = r.payload.get('page_number', '?')
        txt = r.payload.get('text', '')
        passages.append(f"(Page {p_no}) {txt}")
        print(f" • [Page {p_no}] Score: {r.score:.4f} | Preview: {txt[:120]}...")

    if groq_client:
        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a scientific assistant. Answer concisely using provided context."},
                {"role": "user", "content": f"Context:\n" + "\n\n".join(passages) + f"\n\nQuestion: {query}"}
            ]
        )
        print("\n🤖 Groq Answer:")
        print(resp.choices[0].message.content)


--- [Query 1] What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks? ---
 • [Page 8] Score: 1.0000 | Preview: Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and ...
 • [Page 9] Score: 0.6667 | Preview: Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metri...
 • [Page 10] Score: 0.5000 | Preview: Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23
of WSJ)
Parser
Trai...

🤖 Groq Answer:
The BLEU scores for Transformer (big) are 28.4 for English-to-German and 41.8 for English-to-French, according to Table 2.

--- [Query 2] What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)? ---
 • [Page 4] Score: 1.0000 | Preview: Scaled Dot-Product Attention
Multi-Head Attention
Figure 2: (left) Scaled